In [16]:
# ============================================================
# H7: LABEL-AWARE HYPERNETWORK PERSONALIZATION
#
# Key idea:
#
# H6:
#   support embeddings
#        ↓
#   mean pooling
#        ↓
#   student context
#        ↓
#   HyperNetwork
#
# H7:
#   support embeddings + labels
#              ↓
#      label-aware encoder
#              ↓
#      class-aware attention
#              ↓
#        student context
#              ↓
#         HyperNetwork
#              ↓
#   personalized classifier
#
# IMPORTANT:
# TEST PROTOCOL IS UNCHANGED.
#
# Test:
#   unseen student
#        ↓
#   first K real clips (early)
#   OR K random real clips (random)
#        ↓
#   personalization
#        ↓
#   remaining real clips
#
# No artificial balancing is performed on test support.
# ============================================================

import argparse
import json
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(1024)
]

NUM_CLASSES = 4
INPUT_DIM = 1024


# ------------------------------------------------------------
# Encoder dimensions
# ------------------------------------------------------------

EMBED_DIM = 128
CONTEXT_DIM = 128
HYPER_HIDDEN = 128

# Label embedding lets the model explicitly distinguish:
#
# 0 = Highly Disengage
# 1 = Disengage
# 2 = Engage
# 3 = Highly Engage
#
LABEL_EMBED_DIM = 32


# ------------------------------------------------------------
# H7 attention dimensions
# ------------------------------------------------------------

ATTENTION_HIDDEN = 128


# ------------------------------------------------------------
# HyperNetwork output
# ------------------------------------------------------------

HEAD_PARAM_DIM = (
    NUM_CLASSES * EMBED_DIM
    + NUM_CLASSES
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

DEFAULT_EPISODES = 1500
DEFAULT_SEED = 42

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


# ------------------------------------------------------------
# H7 training episode parameters
# ------------------------------------------------------------

TRAIN_K_VALUES = [5, 10, 20]

# Maximum query examples per episode
QUERY_MAX = 64


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(clip_id: str):

    """
    Extract student/group identifier.

    Expected structure:

        videoX_clipindex_person...

    Example:

        video123_17_person4

    becomes:

        video123_person4
    """

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id)
    )

    if not match:
        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(meta, dict):
            continue

        label = meta.get("label")

        if label not in LABELS:
            continue

        embeds = meta.get("embeds")

        if not isinstance(embeds, list):
            continue

        if len(embeds) != INPUT_DIM:
            continue

        group = parse_group_from_clip_id(
            clip_id
        )

        if group is None:
            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32
        )

        if not np.all(np.isfinite(arr)):
            continue

        clip_index_match = re.search(
            r"_(\d+)_",
            str(clip_id)
        )

        clip_index = (
            int(
                clip_index_match.group(1)
            )
            if clip_index_match
            else 0
        )

        records.append(
            {
                "clip_id": str(clip_id),
                "group": group,
                "label": str(label),
                "label_id": LABEL_TO_ID[
                    str(label)
                ],
                "clip_index": clip_index,
                "embedding": arr,
            }
        )

    if not records:

        raise ValueError(
            f"No valid labelled clips found "
            f"in {json_path}"
        )

    df = pd.DataFrame(records)

    emb_matrix = np.vstack(
        df["embedding"].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=["embedding"]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


def load_group_fold_map(
    fold_csv_path
):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        - set(fold_df.columns)
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            ["group", "fold"]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby("group")["fold"]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            "folds: "
            f"{bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold["group"],
            group_fold["fold"].astype(int),
        )
    )


# ============================================================
# STANDARD TEST SUPPORT / QUERY SAMPLING
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):

    """
    TEST SAMPLING.

    This function deliberately remains unchanged from
    the original protocol.

    early:
        first K real clips

    random:
        K random real clips

    query:
        every remaining real clip

    NO class balancing is performed here.
    """

    ordered = (
        student_df
        .sort_values("clip_index")
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            f"insufficient_clips_n="
            f"{len(ordered)}_k={k_shot}",
        )

    if protocol == "early":

        support = ordered.iloc[
            :k_shot
        ].copy()

    elif protocol == "random":

        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = ordered.loc[
            sampled_idx
        ].copy()

    else:

        return (
            None,
            None,
            f"unknown_protocol_{protocol}",
        )

    query = ordered.loc[
        ~ordered.index.isin(
            support.index
        )
    ].copy()

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# H7 CLASS-AWARE TRAINING SUPPORT SAMPLER
# ============================================================

def sample_balanced_support(
    student_df,
    k_shot,
    rng,
):
    """
    H7 TRAINING ONLY.

    Construct a minority-aware support set.

    We DO NOT use this for evaluation.

    Strategy:
        1. Find classes available for the student.
        2. Allocate support slots as evenly as possible.
        3. Sample without replacement.
        4. If a class has too few examples, use all available
           examples and redistribute remaining slots.

    No synthetic data is generated.
    """

    available_classes = []

    class_indices = {}

    for class_id in range(NUM_CLASSES):

        idx = (
            student_df[
                student_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if len(available_classes) == 0:
        return None

    # --------------------------------------------------------
    # Start with as-even-as-possible allocation
    # --------------------------------------------------------

    n_classes = len(
        available_classes
    )

    base = k_shot // n_classes

    remainder = k_shot % n_classes

    allocation = {
        c: base
        for c in available_classes
    }

    # Randomize which classes receive
    # the remainder to avoid systematic bias.
    shuffled_classes = (
        available_classes.copy()
    )

    rng.shuffle(
        shuffled_classes
    )

    for c in shuffled_classes[
        :remainder
    ]:

        allocation[c] += 1

    # --------------------------------------------------------
    # Cap allocation by available examples
    # --------------------------------------------------------

    selected_indices = []

    unused_slots = 0

    for c in available_classes:

        requested = allocation[c]

        available = class_indices[c]

        take = min(
            requested,
            len(available)
        )

        if take < requested:

            unused_slots += (
                requested - take
            )

        chosen = rng.choice(
            available,
            size=take,
            replace=False,
        )

        selected_indices.extend(
            chosen.tolist()
        )

    # --------------------------------------------------------
    # Redistribute unused slots
    # --------------------------------------------------------

    if unused_slots > 0:

        remaining_candidates = []

        selected_set = set(
            selected_indices
        )

        for c in available_classes:

            for idx in class_indices[c]:

                if idx not in selected_set:

                    remaining_candidates.append(
                        idx
                    )

        if remaining_candidates:

            take = min(
                unused_slots,
                len(
                    remaining_candidates
                )
            )

            extra = rng.choice(
                np.asarray(
                    remaining_candidates
                ),
                size=take,
                replace=False,
            )

            selected_indices.extend(
                extra.tolist()
            )

    if len(selected_indices) == 0:
        return None

    return student_df.loc[
        selected_indices
    ].copy()


# ============================================================
# H7 CLASS-AWARE TRAINING QUERY SAMPLER
# ============================================================

def sample_balanced_query(
    query_df,
    max_samples,
    rng,
):
    """
    H7 TRAINING ONLY.

    Construct a class-balanced query batch.

    This ensures minority classes actually contribute to
    the meta-training signal.

    If some classes are unavailable, the available classes
    are balanced as evenly as possible.
    """

    if query_df.empty:
        return query_df

    available_classes = []

    class_indices = {}

    for class_id in range(NUM_CLASSES):

        idx = (
            query_df[
                query_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            available_classes.append(
                class_id
            )

            class_indices[
                class_id
            ] = idx

    if len(available_classes) == 0:
        return query_df.iloc[0:0]

    n_classes = len(
        available_classes
    )

    per_class = max_samples // n_classes

    remainder = max_samples % n_classes

    selected = []

    for c in available_classes:

        idx = class_indices[c]

        take = min(
            per_class,
            len(idx)
        )

        if take > 0:

            chosen = rng.choice(
                idx,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # Fill remaining slots from unused examples.
    selected_set = set(selected)

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(idx)

    remaining_slots = (
        max_samples
        - len(selected)
    )

    if (
        remaining_slots > 0
        and remaining
    ):

        take = min(
            remaining_slots,
            len(remaining)
        )

        extra = rng.choice(
            np.asarray(remaining),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    return query_df.loc[
        selected
    ].copy()


# ============================================================
# LABEL-AWARE SUPPORT ENCODER
# ============================================================

class LabelAwareContextEncoder(
    nn.Module
):
    """
    H7 support encoder.

    Each support example consists of:

        embedding
            +
        explicit class label

    The label is converted to a learned embedding.

    Example:

        visual embedding
             +
        "Highly Disengage"
             ↓
        support representation

    This is fundamentally different from simply averaging
    unlabeled support embeddings.
    """

    def __init__(
        self,
        input_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
        context_dim=CONTEXT_DIM,
        label_embed_dim=LABEL_EMBED_DIM,
        attention_hidden=ATTENTION_HIDDEN,
    ):
        super().__init__()

        # ----------------------------------------------------
        # Clip embedding encoder
        # ----------------------------------------------------

        self.embedding_encoder = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(256),

            nn.Linear(
                256,
                embed_dim
            ),

            nn.ReLU(),

            nn.LayerNorm(
                embed_dim
            ),
        )

        # ----------------------------------------------------
        # Explicit class embedding
        # ----------------------------------------------------

        self.label_embedding = (
            nn.Embedding(
                NUM_CLASSES,
                label_embed_dim
            )
        )

        # ----------------------------------------------------
        # Combine clip + label
        # ----------------------------------------------------

        self.support_fusion = nn.Sequential(

            nn.Linear(
                embed_dim
                + label_embed_dim,
                context_dim
            ),

            nn.ReLU(),

            nn.LayerNorm(
                context_dim
            ),
        )

        # ----------------------------------------------------
        # Attention mechanism
        # ----------------------------------------------------

        self.attention = nn.Sequential(

            nn.Linear(
                context_dim,
                attention_hidden
            ),

            nn.Tanh(),

            nn.Linear(
                attention_hidden,
                1
            ),
        )

        # ----------------------------------------------------
        # Final context projection
        # ----------------------------------------------------

        self.context_projection = nn.Sequential(

            nn.Linear(
                context_dim,
                context_dim
            ),

            nn.ReLU(),

            nn.LayerNorm(
                context_dim
            ),
        )

    def forward(
        self,
        support_x,
        support_y,
    ):

        # ----------------------------------------------------
        # Encode support embeddings
        # ----------------------------------------------------

        visual = (
            self.embedding_encoder(
                support_x
            )
        )

        # ----------------------------------------------------
        # Encode labels explicitly
        # ----------------------------------------------------

        labels = (
            self.label_embedding(
                support_y
            )
        )

        # ----------------------------------------------------
        # Concatenate
        # ----------------------------------------------------

        fused = torch.cat(
            [
                visual,
                labels,
            ],
            dim=1,
        )

        # ----------------------------------------------------
        # Support-level representations
        # ----------------------------------------------------

        support_repr = (
            self.support_fusion(
                fused
            )
        )

        # ----------------------------------------------------
        # Attention over support examples
        # ----------------------------------------------------

        attention_logits = (
            self.attention(
                support_repr
            )
        )

        attention_weights = (
            F.softmax(
                attention_logits,
                dim=0,
            )
        )

        # ----------------------------------------------------
        # Weighted aggregation
        # ----------------------------------------------------

        context = (
            attention_weights
            * support_repr
        ).sum(dim=0)

        # ----------------------------------------------------
        # Final student context
        # ----------------------------------------------------

        context = (
            self.context_projection(
                context
            )
        )

        return (
            context,
            attention_weights.squeeze(-1),
        )


# ============================================================
# HYPERNETWORK
# ============================================================

class HyperNetwork(
    nn.Module
):
    """
    Generates a personalized linear classifier from
    the label-aware student context.
    """

    def __init__(
        self,
        context_dim=CONTEXT_DIM,
        hidden_dim=HYPER_HIDDEN,
        embed_dim=EMBED_DIM,
    ):
        super().__init__()

        self.embed_dim = embed_dim

        self.net = nn.Sequential(

            nn.Linear(
                context_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.ReLU(),
        )

        self.output = nn.Linear(
            hidden_dim,
            HEAD_PARAM_DIM,
        )

    def forward(
        self,
        context
    ):

        params = self.output(
            self.net(context)
        )

        weight_end = (
            NUM_CLASSES
            * self.embed_dim
        )

        weight = (
            params[
                :weight_end
            ]
            .view(
                NUM_CLASSES,
                self.embed_dim,
            )
        )

        bias = params[
            weight_end:
        ]

        return (
            weight,
            bias,
        )


# ============================================================
# QUERY ENCODER
# ============================================================

class QueryEncoder(
    nn.Module
):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        embed_dim=EMBED_DIM,
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                256
            ),

            nn.ReLU(),

            nn.LayerNorm(256),

            nn.Linear(
                256,
                embed_dim
            ),

            nn.ReLU(),

            nn.LayerNorm(
                embed_dim
            ),
        )

    def forward(
        self,
        x
    ):

        return self.net(x)


# ============================================================
# FULL H7 MODEL
# ============================================================

class H7HyperNetworkModel(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.context_encoder = (
            LabelAwareContextEncoder()
        )

        self.query_encoder = (
            QueryEncoder()
        )

        self.hypernetwork = (
            HyperNetwork()
        )

    def generate_student_head(
        self,
        support_x,
        support_y,
    ):

        context, attention = (
            self.context_encoder(
                support_x,
                support_y,
            )
        )

        weight, bias = (
            self.hypernetwork(
                context
            )
        )

        return (
            weight,
            bias,
            context,
            attention,
        )

    def predict(
        self,
        support_x,
        support_y,
        query_x,
    ):

        (
            weight,
            bias,
            context,
            attention,
        ) = self.generate_student_head(
            support_x,
            support_y,
        )

        query_emb = (
            self.query_encoder(
                query_x
            )
        )

        logits = F.linear(
            query_emb,
            weight,
            bias,
        )

        return (
            logits,
            context,
            attention,
        )


# ============================================================
# STANDARD CROSS ENTROPY
# ============================================================

def classification_loss(
    logits,
    targets,
):

    return F.cross_entropy(
        logits,
        targets,
    )


# ============================================================
# EPISODIC META-TRAINING
# ============================================================

def episodic_train(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):
    """
    H7 meta-training.

    Important:

    Training episodes are class-aware.

    Evaluation episodes are NOT.

    This is intentional.

    H7 asks:

        Does explicitly exposing the meta-learner to
        minority-class evidence during training, while
        also giving the HyperNetwork support labels,
        improve unseen-student adaptation?
    """

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df["group"].unique()
    )

    if len(groups) == 0:

        raise ValueError(
            "No training students available."
        )

    successful_episodes = 0

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Sample student/task
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = (
            train_df[
                train_df["group"]
                == group
            ]
            .copy()
        )

        # ----------------------------------------------------
        # Random K
        # ----------------------------------------------------

        k = int(
            rng.choice(
                TRAIN_K_VALUES
            )
        )

        if len(student_df) <= k:

            continue

        # ----------------------------------------------------
        # H7:
        # class-balanced support
        # ----------------------------------------------------

        support_df = (
            sample_balanced_support(
                student_df,
                k,
                rng,
            )
        )

        if (
            support_df is None
            or len(support_df) == 0
        ):

            continue

        # ----------------------------------------------------
        # Query = everything not used for support
        # ----------------------------------------------------

        query_df = student_df.loc[
            ~student_df.index.isin(
                support_df.index
            )
        ].copy()

        if query_df.empty:
            continue

        # ----------------------------------------------------
        # H7:
        # class-balanced query batch
        # ----------------------------------------------------

        query_df = (
            sample_balanced_query(
                query_df,
                QUERY_MAX,
                rng,
            )
        )

        if query_df.empty:
            continue

        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        support_x_np = scaler.transform(
            support_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_x_np = scaler.transform(
            query_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        support_y_np = (
            support_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        query_y_np = (
            query_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        support_x = torch.tensor(
            support_x_np,
            dtype=torch.float32,
            device=device,
        )

        support_y = torch.tensor(
            support_y_np,
            dtype=torch.long,
            device=device,
        )

        query_x = torch.tensor(
            query_x_np,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y_np,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        (
            logits,
            context,
            attention,
        ) = model.predict(
            support_x,
            support_y,
            query_x,
        )

        # ----------------------------------------------------
        # H7 uses standard CE
        #
        # This deliberately isolates the architectural
        # contribution of label-aware personalization.
        # ----------------------------------------------------

        loss = classification_loss(
            logits,
            query_y,
        )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        successful_episodes += 1

        if (
            (episode + 1) % 100
            == 0
        ):

            print(
                f"    Episode "
                f"{episode + 1}/{episodes} "
                f"loss={loss.item():.4f} "
                f"successful="
                f"{successful_episodes}"
            )

    print(
        f"    Successful training "
        f"episodes: "
        f"{successful_episodes}/"
        f"{episodes}"
    )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {
        "accuracy": float(
            accuracy
        ),

        "balanced_accuracy": float(
            balanced_accuracy
        ),

        "macro_f1": float(
            macro_f1
        ),

        "precision": precision,

        "recall": recall,

        "f1": f1,

        "support": support,

        "confusion_matrix": cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query,
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get,
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# EVALUATION
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    support_x_np = scaler.transform(
        support_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    query_x_np = scaler.transform(
        query_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    support_y_np = (
        support_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_query = (
        query_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    support_x = torch.tensor(
        support_x_np,
        dtype=torch.float32,
        device=device,
    )

    support_y = torch.tensor(
        support_y_np,
        dtype=torch.long,
        device=device,
    )

    query_x = torch.tensor(
        query_x_np,
        dtype=torch.float32,
        device=device,
    )

    model.eval()

    with torch.no_grad():

        (
            logits,
            context,
            attention,
        ) = model.predict(
            support_x,
            support_y,
            query_x,
        )

        y_pred = (
            torch.argmax(
                logits,
                dim=1,
            )
            .cpu()
            .numpy()
        )

    metrics = calculate_metrics(
        y_query,
        y_pred,
    )

    majority_pred = (
        majority_prediction(
            y_query
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_query,
            majority_pred,
        )
    )

    return (
        y_query,
        y_pred,
        metrics,
        majority_metrics,
    )


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df["group"].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []
    per_student_rows = []
    per_class_rows = []
    confusion_rows = []
    skipped_rows = []

    # --------------------------------------------------------
    # Outer 5-fold evaluation
    # --------------------------------------------------------

    fold_ids = sorted(
        set(
            group_fold.values()
        )
    )

    for fold_id in fold_ids:

        print()
        print("=" * 80)
        print(
            f"H7 OUTER FOLD {fold_id}"
        )
        print("=" * 80)

        test_groups = [
            g
            for g, f in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df["group"].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df["group"].isin(
                test_groups
            )
        ].copy()

        if train_df.empty:

            raise ValueError(
                f"No training clips "
                f"for fold {fold_id}"
            )

        # ----------------------------------------------------
        # Leakage check
        # ----------------------------------------------------

        overlap = (
            set(train_df["group"])
            &
            set(test_df["group"])
        )

        if overlap:

            raise RuntimeError(
                f"DATA LEAKAGE in fold "
                f"{fold_id}: "
                f"{overlap}"
            )

        print(
            f"Training students: "
            f"{train_df['group'].nunique()}"
        )

        print(
            f"Test students: "
            f"{test_df['group'].nunique()}"
        )

        # ----------------------------------------------------
        # Training class distribution
        # ----------------------------------------------------

        print()
        print(
            "Training class distribution:"
        )

        print(
            train_df[
                "label"
            ].value_counts()
        )

        # ----------------------------------------------------
        # Fit scaler ONLY on training students
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train = (
            train_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        scaler.fit(
            X_train
        )

        # ----------------------------------------------------
        # Fresh H7 model
        # ----------------------------------------------------

        set_seed(
            seed + fold_id
        )

        model = (
            H7HyperNetworkModel()
        )

        model.to(device)

        print()
        print(
            f"H7 meta-training for "
            f"{episodes} episodes..."
        )

        episodic_train(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                + fold_id * 1000
            ),
            device=device,
        )

        # ----------------------------------------------------
        # Test students
        # ----------------------------------------------------

        for group in test_groups:

            student_df = test_df[
                test_df["group"]
                == group
            ].copy()

            if student_df.empty:
                continue

            for k in k_values:

                for protocol in [
                    "random",
                    "early",
                ]:

                    local_seed = (
                        seed
                        + fold_id * 10000
                        + k * 100
                        + sum(
                            ord(ch)
                            for ch in group
                        )
                        + (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = (
                        sample_support_query(
                            student_df,
                            k,
                            protocol,
                            rng,
                        )
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {
                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "reason":
                                    reason,
                            }
                        )

                        continue

                    # ------------------------------------------------
                    # Evaluate using REAL support.
                    #
                    # No class balancing.
                    # No synthetic examples.
                    # No modification.
                    # ------------------------------------------------

                    (
                        y_true,
                        y_pred,
                        metrics,
                        majority_metrics,
                    ) = evaluate_student(
                        model,
                        support_df,
                        query_df,
                        scaler,
                        device,
                    )

                    row = {

                        "method":
                            "H7_label_aware_hypernetwork",

                        "outer_fold":
                            fold_id,

                        "student_group":
                            group,

                        "k_shot":
                            k,

                        "support_protocol":
                            protocol,

                        "support_size":
                            len(support_df),

                        "query_size":
                            len(query_df),

                        "accuracy":
                            metrics[
                                "accuracy"
                            ],

                        "balanced_accuracy":
                            metrics[
                                "balanced_accuracy"
                            ],

                        "macro_f1":
                            metrics[
                                "macro_f1"
                            ],

                        "majority_accuracy":
                            majority_metrics[
                                "accuracy"
                            ],

                        "majority_balanced_accuracy":
                            majority_metrics[
                                "balanced_accuracy"
                            ],

                        "majority_macro_f1":
                            majority_metrics[
                                "macro_f1"
                            ],
                    }

                    per_student_rows.append(
                        row
                    )

                    # ------------------------------------------------
                    # Per-class
                    # ------------------------------------------------

                    for class_id, label in (
                        enumerate(LABELS)
                    ):

                        per_class_rows.append(
                            {

                                "method":
                                    "H7_label_aware_hypernetwork",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][class_id]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][class_id]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][class_id]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][class_id]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {

                                    "method":
                                        "H7_label_aware_hypernetwork",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = (
        pd.DataFrame(
            per_student_rows
        )
    )

    per_class_df = (
        pd.DataFrame(
            per_class_rows
        )
    )

    confusion_df = (
        pd.DataFrame(
            confusion_rows
        )
    )

    skipped_df = (
        pd.DataFrame(
            skipped_rows
        )
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No student evaluations "
            "were produced."
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # OUTPUT DIRECTORY
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "h7_per_student.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "h7_summary.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "h7_per_class.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "h7_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "h7_skipped.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    per_class_df.to_csv(
        per_class_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    # ========================================================
    # PRINT SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H7: LABEL-AWARE HYPERNETWORK RESULTS"
    )
    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    # ========================================================
    # PER CLASS
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H7 PER-CLASS RESULTS"
    )
    print("=" * 80)

    class_summary = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    print(
        class_summary.to_string(
            index=False
        )
    )

    # ========================================================
    # SKIPPED
    # ========================================================

    print()
    print("=" * 80)
    print(
        "H7 SKIPPED CASES"
    )
    print("=" * 80)

    if skipped_df.empty:

        print("None.")

    else:

        print(
            skipped_df[
                "reason"
            ].value_counts()
        )

    # ========================================================
    # SAVED FILES
    # ========================================================

    print()
    print("Saved:")

    print(
        per_student_path
    )

    print(
        summary_path
    )

    print(
        per_class_path
    )

    print(
        confusion_path
    )

    print(
        skipped_path
    )

    return (
        per_student_df,
        summary_df,
        per_class_df,
        confusion_df,
        skipped_df,
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser(
        description=(
            "H7 Label-Aware HyperNetwork "
            "for few-shot unseen-student "
            "engagement classification."
        )
    )

    parser.add_argument(
        "--json",
        type=str,
        default=(
            "/content/final_data_1.json"
        ),
    )

    parser.add_argument(
        "--folds",
        type=str,
        default=(
            "/content/"
            "cmose_h1_h2_fold_protocol.csv"
        ),
    )

    parser.add_argument(
        "--k-values",
        type=str,
        default="5,10,20",
    )

    parser.add_argument(
        "--episodes",
        type=int,
        default=DEFAULT_EPISODES,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=DEFAULT_SEED,
    )

    parser.add_argument(
        "--output-dir",
        type=str,
        default="/content/h7_results",
    )

    parser.add_argument(
        "--device",
        type=str,
        default="auto",
        choices=[
            "auto",
            "cpu",
            "cuda",
        ],
    )

    args = parser.parse_args()

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if args.device == "cuda":

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA requested but no CUDA "
                "device is available."
            )

        device = torch.device(
            "cuda"
        )

    elif args.device == "cpu":

        device = torch.device(
            "cpu"
        )

    else:

        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(
        f"Using device: {device}"
    )

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed(
        args.seed
    )

    # --------------------------------------------------------
    # K
    # --------------------------------------------------------

    k_values = [
        int(v.strip())
        for v in args.k_values.split(",")
        if v.strip()
    ]

    if not k_values:

        raise ValueError(
            "No K values supplied."
        )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            args.json
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    # --------------------------------------------------------
    # Fold protocol
    # --------------------------------------------------------

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            args.folds
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # --------------------------------------------------------
    # Run
    # --------------------------------------------------------

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=k_values,
        seed=args.seed,
        episodes=args.episodes,
        device=device,
        output_dir=args.output_dir,
    )


# ============================================================
# COLAB ENTRY POINT
#
# IMPORTANT:
# We do NOT use argparse directly when running inside a
# notebook because Colab injects:
#
#     -f /root/.local/share/jupyter/runtime/...
#
# which caused your previous error.
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Direct Colab configuration
    # --------------------------------------------------------

    JSON_PATH = (
        "/content/final_data_1.json"
    )

    FOLDS_PATH = (
        "/content/"
        "cmose_h1_h2_fold_protocol.csv"
    )

    OUTPUT_DIR = (
        "/content/h7_results"
    )

    K_VALUES = [5, 10, 20]

    EPISODES = 1500

    SEED = 42

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device(
            "cuda"
        )

        print(
            "CUDA available: True"
        )

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    else:

        DEVICE = torch.device(
            "cpu"
        )

        print(
            "CUDA available: False"
        )

        print(
            "Using CPU."
        )

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed(
        SEED
    )

    # --------------------------------------------------------
    # Load
    # --------------------------------------------------------

    print()
    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            JSON_PATH
        )
    )

    print(
        f"Loaded "
        f"{len(clip_df)} labelled clips "
        f"from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            FOLDS_PATH
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # --------------------------------------------------------
    # Run H7
    # --------------------------------------------------------

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=K_VALUES,
        seed=SEED,
        episodes=EPISODES,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )

CUDA available: True
GPU: Tesla T4

Loading clip data...
Loaded 11902 labelled clips from 103 students.

Class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Loading fold protocol...
Fold protocol contains 82 students.

H7 OUTER FOLD 1
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

H7 meta-training for 1500 episodes...
    Episode 100/1500 loss=0.9543 successful=100
    Episode 200/1500 loss=0.9497 successful=200
    Episode 300/1500 loss=0.5012 successful=300
    Episode 400/1500 loss=0.4883 successful=400
    Episode 500/1500 loss=0.4432 successful=500
    Episode 600/1500 loss=0.0256 successful=600
    Episode 700/1500 loss=0.8914 successful=700
    Episode 800/1500 loss=0.4762 successful=800
    Episode 900/1500 loss=0.2043 succes

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H7 OUTER FOLD 2
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

H7 meta-training for 1500 episodes...
    Episode 100/1500 loss=1.2262 successful=100
    Episode 200/1500 loss=0.7521 successful=200
    Episode 300/1500 loss=0.6056 successful=300
    Episode 400/1500 loss=0.2201 successful=400
    Episode 500/1500 loss=0.3532 successful=500
    Episode 600/1500 loss=0.3668 successful=600
    Episode 700/1500 loss=0.2086 successful=700
    Episode 800/1500 loss=0.3070 successful=800
    Episode 900/1500 loss=0.0838 successful=900
    Episode 1000/1500 loss=0.5529 successful=1000
    Episode 1100/1500 loss=0.5664 successful=1100
    Episode 1200/1500 loss=0.7501 successful=1200
    Episode 1300/1500 loss=0.1945 successful=1300
    Episode 1400/1500 loss=0.1576 successful=1400
    Episode 1500/1500 loss=0.1334 successful=1500
    Succes

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H7 OUTER FOLD 3
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

H7 meta-training for 1500 episodes...
    Episode 100/1500 loss=0.7702 successful=100
    Episode 200/1500 loss=0.6002 successful=200
    Episode 300/1500 loss=0.6052 successful=300
    Episode 400/1500 loss=0.5927 successful=400
    Episode 500/1500 loss=0.7015 successful=500
    Episode 600/1500 loss=0.4392 successful=600
    Episode 700/1500 loss=0.7319 successful=700
    Episode 800/1500 loss=0.1354 successful=800
    Episode 900/1500 loss=0.6917 successful=900
    Episode 1000/1500 loss=0.5200 successful=1000
    Episode 1100/1500 loss=0.3084 successful=1100
    Episode 1200/1500 loss=0.7256 successful=1200
    Episode 1300/1500 loss=0.1360 successful=1300
    Episode 1400/1500 loss=0.7477 successful=1400
    Episode 1500/1500 loss=0.3156 successful=1500
    Succes

...


H7 OUTER FOLD 4
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

H7 meta-training for 1500 episodes...
    Episode 100/1500 loss=0.7702 successful=100
    Episode 200/1500 loss=0.1627 successful=200
    Episode 300/1500 loss=0.6858 successful=300
    Episode 400/1500 loss=0.3105 successful=400
    Episode 500/1500 loss=0.7607 successful=500
    Episode 600/1500 loss=0.3381 successful=600
    Episode 700/1500 loss=0.8459 successful=700
    Episode 800/1500 loss=0.7057 successful=800
    Episode 900/1500 loss=0.1568 successful=900
    Episode 1000/1500 loss=0.5870 successful=1000
    Episode 1100/1500 loss=0.0871 successful=1100
    Episode 1200/1500 loss=0.6427 successful=1200
    Episode 1300/1500 loss=0.3363 successful=1300
    Episode 1400/1500 loss=0.4593 successful=1400
    Episode 1500/1500 loss=0.7791 successful=1500
    Succes

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H7 OUTER FOLD 5
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

H7 meta-training for 1500 episodes...
    Episode 100/1500 loss=0.7866 successful=100
    Episode 200/1500 loss=0.3575 successful=200
    Episode 300/1500 loss=0.7284 successful=300
    Episode 400/1500 loss=0.8987 successful=400
    Episode 500/1500 loss=0.3402 successful=500
    Episode 600/1500 loss=1.2128 successful=600
    Episode 700/1500 loss=0.6088 successful=700
    Episode 800/1500 loss=0.5543 successful=800
    Episode 900/1500 loss=0.5307 successful=900
    Episode 1000/1500 loss=0.5228 successful=1000
    Episode 1100/1500 loss=0.0497 successful=1100
    Episode 1200/1500 loss=0.2723 successful=1200
    Episode 1300/1500 loss=0.2511 successful=1300
    Episode 1400/1500 loss=0.4790 successful=1400
    Episode 1500/1500 loss=0.1950 successful=1500
    Succes

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


H7: LABEL-AWARE HYPERNETWORK RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1
           early       5          82              11016       0.619943     0.161587                0.455096              0.152541       0.326224     0.088749                0.708270                         0.307927                0.205566
           early      10          82              10606       0.621602     0.161899                0.458538              0.152938       0.322959     0.089916                0.710367                         0.315041                0.205928
           early      20          82               9786       0.622301     0.168831                0.464378              0.163483       0.318088     0.093192                0.713848                         0.325203                0.206457
      